In [1]:
# Cell 1 — Download semiconductor stock data (Oct 2025 → last news date)

import pandas as pd

NEWS_PATH = "Data/labelled_semiconductor_news_vader.csv"

news = pd.read_csv(NEWS_PATH)
news["date"] = pd.to_datetime(news["date"])

start_date = "2025-10-01"
end_date = news["date"].max().strftime("%Y-%m-%d")

print("News date range:", news["date"].min().date(), "→", news["date"].max().date())
print("Downloading prices:", start_date, "→", end_date)

tickers = [
    "NVDA","AMD","INTC","TSM","AVGO","QCOM","MU","TXN","ASML",
    "AMAT","LRCX","KLAC","ADI","NXPI","MRVL","ON","MCHP","TER"
]
benchmarks = ["SOXX", "SMH"]
tickers_all = tickers + benchmarks

try:
    import yfinance as yf
except ImportError:
    raise ImportError("yfinance not installed. Run: pip install yfinance")

end_plus_one = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

prices = yf.download(
    tickers_all,
    start=start_date,
    end=end_plus_one,
    auto_adjust=True,
    group_by="ticker",
    progress=False
)

downloaded = sorted(set(prices.columns.get_level_values(0)))
missing = sorted(set(tickers_all) - set(downloaded))
print(f"Downloaded tickers: {len(downloaded)}")
if missing:
    print("Missing tickers (no data returned):", missing)

rows = []
for t in tickers_all:
    if t in prices.columns.get_level_values(0):
        df_t = prices[t].copy()
        df_t.columns = [c.lower() for c in df_t.columns]
        df_t = df_t.reset_index()
        if "Date" in df_t.columns:
            df_t = df_t.rename(columns={"Date": "date"})
        elif "index" in df_t.columns:
            df_t = df_t.rename(columns={"index": "date"})
        df_t["ticker"] = t
        rows.append(df_t)

price_long = pd.concat(rows, ignore_index=True)
keep_cols = [c for c in ["date","ticker","open","high","low","close","volume"] if c in price_long.columns]
price_long = price_long[keep_cols].sort_values(["ticker","date"])

out_long = "Data/semiconductor_prices_oct2025_long.csv"
price_long.to_csv(out_long, index=False)

print("Saved:", out_long)
print(price_long.head())

News date range: 2025-11-01 → 2025-12-31
Downloaded tickers: 20
Saved: Data/semiconductor_prices_oct2025_long.csv
          date ticker        open        high         low       close   volume
768 2025-10-01    ADI  242.852283  244.725656  235.388638  238.437866  4916300
769 2025-10-02    ADI  241.277830  243.111351  239.593776  240.819443  2774100
770 2025-10-03    ADI  241.198107  245.333495  240.739720  241.138321  2118500
771 2025-10-06    ADI  244.147691  244.147691  238.079128  241.646530  4021200
772 2025-10-07    ADI  242.533394  242.533394  232.209855  232.927322  3397800


In [2]:
# Cell 2 — Add forward returns (1d, 3d, 5d)

import numpy as np
import pandas as pd

price_long = pd.read_csv("Data/semiconductor_prices_oct2025_long.csv")
price_long["date"] = pd.to_datetime(price_long["date"])
price_long = price_long.sort_values(["ticker", "date"]).reset_index(drop=True)

def add_forward_returns(df, horizons=(1,3,5)):
    df = df.copy()
    for h in horizons:
        df[f"ret_fwd_{h}d"] = df["close"].shift(-h) / df["close"] - 1
    return df

price_with_rets = (
    price_long.groupby("ticker", group_keys=False)
    .apply(add_forward_returns, horizons=(1,3,5))
)

out_path = "Data/semiconductor_prices_oct2025_with_returns.csv"
price_with_rets.to_csv(out_path, index=False)

print("Saved:", out_path)
print(price_with_rets.head(10))

Saved: Data/semiconductor_prices_oct2025_with_returns.csv
        date ticker        open        high         low       close   volume  \
0 2025-10-01    ADI  242.852283  244.725656  235.388638  238.437866  4916300   
1 2025-10-02    ADI  241.277830  243.111351  239.593776  240.819443  2774100   
2 2025-10-03    ADI  241.198107  245.333495  240.739720  241.138321  2118500   
3 2025-10-06    ADI  244.147691  244.147691  238.079128  241.646530  4021200   
4 2025-10-07    ADI  242.533394  242.533394  232.209855  232.927322  3397800   
5 2025-10-08    ADI  232.977151  238.517581  232.977151  237.092606  4017900   
6 2025-10-09    ADI  236.006437  237.550985  234.272571  237.042786  3069200   
7 2025-10-10    ADI  236.903279  238.198691  223.919139  224.526993  4425100   
8 2025-10-13    ADI  227.516443  234.990047  226.161229  233.844086  4609900   
9 2025-10-14    ADI  230.336461  237.939612  228.323581  234.571503  3431200   

   ret_fwd_1d  ret_fwd_3d  ret_fwd_5d  
0    0.009988    0.01

/var/folders/5_/c20qy50s35jdz3bv32kv3lpw0000gn/T/ipykernel_75465/1530850809.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_forward_returns, horizons=(1,3,5))


In [3]:
# Cell 3 — Load labelled news + binary label + date split

import pandas as pd
import numpy as np

NEWS_PATH = "Data/labelled_semiconductor_news_vader.csv"
TEXT_COL = "_vader_text"
LABEL_COL = "vader_label_num"

news = pd.read_csv(NEWS_PATH)

# Ensure date is present in the same DF we filter/split
news["date"] = pd.to_datetime(news["date"])

news = news[[TEXT_COL, LABEL_COL, "date"]].dropna()

# Binary label: positive vs non-positive
news["label"] = (news[LABEL_COL] == 1).astype(int)

print(news["label"].value_counts())

train = news[news["date"] < "2025-12-01"].copy()
val   = news[(news["date"] >= "2025-12-01") & (news["date"] < "2025-12-15")].copy()
test  = news[news["date"] >= "2025-12-15"].copy()

print("Split sizes:", len(train), len(val), len(test))
train.head()

label
0    157
1    135
Name: count, dtype: int64
Split sizes: 123 71 98


,_vader_text,vader_label_num,date,label
0,ai chipmaker nvidia hits record $5 trillion ma...,0,2025-11-01,0
1,microsoft ceo doesnt want to buy nvidia ai gpu...,1,2025-11-01,1
2,nvidia ( nvda ) ceo is confident chinese milit...,1,2025-11-01,1
3,"nvidia news today : korea gpu build - out , sa...",0,2025-11-01,0
4,1 super semiconductor stock to buy hand over f...,1,2025-11-02,1


In [4]:
# Cell 4 — Build vocab + encode to fixed MAX_LEN

from collections import Counter

MAX_VOCAB = 20000
MAX_LEN = 40

def tokenize(text):
    return str(text).split()

counter = Counter()
for t in train[TEXT_COL]:
    counter.update(tokenize(t))

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

X_train = np.array([encode(t) for t in train[TEXT_COL]])
X_val   = np.array([encode(t) for t in val[TEXT_COL]])
X_test  = np.array([encode(t) for t in test[TEXT_COL]])

y_train = train["label"].values.astype(np.float32)
y_val   = val["label"].values.astype(np.float32)
y_test  = test["label"].values.astype(np.float32)

print("X_train shape:", X_train.shape, "| y_train:", y_train.shape)
print("Vocab size:", len(vocab))

X_train shape: (123, 40) | y_train: (123,)
Vocab size: 886


In [5]:
# Cell 5 — Torch Dataset + DataLoaders

import torch
from torch.utils.data import Dataset, DataLoader

class NewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32

train_loader = DataLoader(NewsDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NewsDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(NewsDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

len(train_loader), len(val_loader), len(test_loader)

(4, 3, 4)

In [6]:
# Cell 6 — Transformer model definition (replaces BiLSTM)

import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_len: int,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.2,
        pad_idx: int = 0,
    ):
        super().__init__()
        assert d_model % nhead == 0, "d_model must be divisible by nhead"

        self.pad_idx = pad_idx
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc = PositionalEncoding(d_model=d_model, max_len=max_len, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        key_padding_mask = (x == self.pad_idx)  # True for PAD
        h = self.embedding(x) * math.sqrt(self.d_model)
        h = self.pos_enc(h)
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)

        mask = (~key_padding_mask).unsqueeze(-1).float()
        h = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        h = self.dropout(h)
        return self.fc(h).squeeze(1)  # logits

In [7]:
# Cell 7 — Train baseline Transformer + validate

import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.2

model = TransformerClassifier(
    vocab_size=len(vocab),
    max_len=MAX_LEN,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=FF_DIM,
    dropout=DROPOUT,
    pad_idx=0
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(loader):
    model.train()
    total_loss = 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

@torch.no_grad()
def eval_epoch(loader):
    model.eval()
    preds, labels = [], []
    for X, y in loader:
        X = X.to(device)
        logits = model(X)
        preds.extend(torch.sigmoid(logits).cpu().numpy())
        labels.extend(y.numpy())
    return np.array(preds), np.array(labels)

EPOCHS = 5
for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader)
    val_preds, val_labels = eval_epoch(val_loader)
    val_acc = ((val_preds > 0.5) == val_labels).mean()
    print(f"Epoch {epoch+1} | Train Loss {train_loss:.4f} | Val Acc {val_acc:.4f}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


Epoch 1 | Train Loss 0.6927 | Val Acc 0.5915
Epoch 2 | Train Loss 0.6367 | Val Acc 0.6620
Epoch 3 | Train Loss 0.5846 | Val Acc 0.6197
Epoch 4 | Train Loss 0.5324 | Val Acc 0.6761
Epoch 5 | Train Loss 0.4484 | Val Acc 0.6338


In [8]:
# Cell 8 — Test metrics

from sklearn.metrics import classification_report, roc_auc_score

test_preds, test_labels = eval_epoch(test_loader)

print(classification_report(test_labels, test_preds > 0.5))
print("ROC-AUC:", roc_auc_score(test_labels, test_preds))

              precision    recall  f1-score   support

         0.0       0.57      0.74      0.64        50
         1.0       0.61      0.42      0.49        48

    accuracy                           0.58        98
   macro avg       0.59      0.58      0.57        98
weighted avg       0.59      0.58      0.57        98

ROC-AUC: 0.5808333333333334


In [9]:
# Cell 9 — Save baseline Transformer model + predictions CSV

import os
import torch

os.makedirs("Models", exist_ok=True)

torch.save(model.state_dict(), "Models/transformer_sentiment.pt")

test_results = test.copy()
test_results["pred_prob"] = test_preds
test_results["pred_label"] = (test_preds > 0.5).astype(int)

test_results.to_csv("Data/transformer_test_predictions.csv", index=False)

print("Saved model: Models/transformer_sentiment.pt")
print("Saved predictions: Data/transformer_test_predictions.csv")

Saved model: Models/transformer_sentiment.pt
Saved predictions: Data/transformer_test_predictions.csv


In [10]:
# Cell 10 — Transformer full param grid search + save best model + save all results
# Uses the same encoded inputs (X_train, y_train, X_val, y_val, X_test, y_test), vocab, MAX_LEN

import os, json, random, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_fscore_support

os.makedirs("Models", exist_ok=True)
os.makedirs("Data", exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class NewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def make_loaders(batch_size):
    train_loader = DataLoader(NewsDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(NewsDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(NewsDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

# --- Model classes (reusing TransformerClassifier) ---
# Assumes you've already run Cell 6 defining PositionalEncoding + TransformerClassifier.
# If not, re-run Cell 6 before running this grid-search cell.

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / max(len(loader), 1)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    probs, labels = [], []
    for Xb, yb in loader:
        Xb = Xb.to(device)
        logits = model(Xb)
        p = torch.sigmoid(logits).cpu().numpy()
        probs.append(p)
        labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def best_threshold_by_f1(y_true, y_prob, thresholds=np.linspace(0.1, 0.9, 33)):
    best_t, best_f1v = 0.5, -1
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1v = f1_score(y_true, y_pred, zero_division=0)
        if f1v > best_f1v:
            best_f1v, best_t = f1v, t
    return float(best_t), float(best_f1v)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    p, r, f1v, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan")
    acc = (y_pred == y_true).mean()
    return {
        "acc": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1v),
        "roc_auc": float(auc),
        "threshold": float(threshold),
    }

param_grid = {
    "d_model":   [64, 128, 256],
    "nhead":     [2, 4, 8],
    "ff_dim":    [128, 256, 512],
    "num_layers":[1, 2, 3],
    "dropout":   [0.1, 0.2, 0.3, 0.4],
    "lr":        [1e-3, 5e-4, 3e-4],
    "batch_size":[16, 32],
    "use_pos_weight": [True, False],
    "epochs":    [50],
    "patience":  [2],
}

EARLY_STOP_ON = "roc_auc"  # or "f1"

def iter_grid(grid):
    keys = list(grid.keys())
    def rec(i, cur):
        if i == len(keys):
            yield cur
            return
        k = keys[i]
        for v in grid[k]:
            nxt = dict(cur); nxt[k] = v
            yield from rec(i+1, nxt)
    yield from rec(0, {})

pos = float(y_train.sum())
neg = float(len(y_train) - pos)
pos_weight_tensor = torch.tensor([neg / max(pos, 1.0)], device=device, dtype=torch.float)

results = []
best = {"score": -1e9, "cfg": None, "state_dict": None, "val_metrics": None}

total_configs = sum(1 for _ in iter_grid(param_grid))
print(f"Grid configs (pre-skip): {total_configs}")

config_id = 0
for cfg in iter_grid(param_grid):
    config_id += 1
    print(f"\n[{config_id}/{total_configs}] cfg={cfg}")

    if cfg["d_model"] % cfg["nhead"] != 0:
        print("  (skip) invalid: d_model must be divisible by nhead")
        continue

    train_loader, val_loader, test_loader = make_loaders(cfg["batch_size"])

    model = TransformerClassifier(
        vocab_size=len(vocab),
        max_len=MAX_LEN,
        d_model=cfg["d_model"],
        nhead=cfg["nhead"],
        num_layers=cfg["num_layers"],
        dim_feedforward=cfg["ff_dim"],
        dropout=cfg["dropout"],
        pad_idx=0
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor) if cfg["use_pos_weight"] else nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

    best_epoch_score = -1e9
    best_epoch_state = None
    bad_epochs = 0

    for epoch in range(1, cfg["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

        val_prob, val_true = predict_probs(model, val_loader)
        t_star, _ = best_threshold_by_f1(val_true, val_prob)
        val_metrics = compute_metrics(val_true, val_prob, threshold=t_star)

        score = val_metrics[EARLY_STOP_ON]
        print(f"  epoch {epoch} | train_loss={train_loss:.4f} | val_auc={val_metrics['roc_auc']:.4f} "
              f"| val_f1={val_metrics['f1']:.4f} | t*={t_star:.2f}")

        if score > best_epoch_score:
            best_epoch_score = score
            best_epoch_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                break

    if best_epoch_state is None:
        print("  (skip) no best state recorded (unexpected)")
        continue

    model.load_state_dict(best_epoch_state)

    val_prob, val_true = predict_probs(model, val_loader)
    t_star, _ = best_threshold_by_f1(val_true, val_prob)
    val_metrics = compute_metrics(val_true, val_prob, threshold=t_star)

    test_prob, test_true = predict_probs(model, test_loader)
    test_metrics = compute_metrics(test_true, test_prob, threshold=t_star)

    row = {
        **cfg,
        "early_stop_on": EARLY_STOP_ON,
        "val_acc": val_metrics["acc"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_threshold": val_metrics["threshold"],
        "test_acc": test_metrics["acc"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_roc_auc": test_metrics["roc_auc"],
    }
    results.append(row)

    if val_metrics[EARLY_STOP_ON] > best["score"]:
        best["score"] = val_metrics[EARLY_STOP_ON]
        best["cfg"] = cfg
        best["state_dict"] = best_epoch_state
        best["val_metrics"] = val_metrics

results_df = pd.DataFrame(results).sort_values(by=f"val_{EARLY_STOP_ON}", ascending=False).reset_index(drop=True)
results_csv = "Data/transformer_grid_results.csv"
results_df.to_csv(results_csv, index=False)
print("\nSaved:", results_csv)
if len(results_df) > 0:
    print("Top 5 configs:\n", results_df.head(5)[
        ["d_model","nhead","ff_dim","num_layers","dropout","lr","batch_size","use_pos_weight",
         f"val_{EARLY_STOP_ON}","val_f1","val_roc_auc","test_f1","test_roc_auc"]
    ])

best_path = "Models/transformer_best.pt"
torch.save(best["state_dict"], best_path)
print("\nBest model saved:", best_path)
print("Best cfg:", best["cfg"])
print("Best val metrics:", best["val_metrics"])

with open("Data/transformer_best_config.json", "w") as f:
    json.dump({"cfg": best["cfg"], "val_metrics": best["val_metrics"], "early_stop_on": EARLY_STOP_ON}, f, indent=2)

print("Saved: Data/transformer_best_config.json")

Grid configs (pre-skip): 3888

[1/3888] cfg={'d_model': 64, 'nhead': 2, 'ff_dim': 128, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 16, 'use_pos_weight': True, 'epochs': 50, 'patience': 2}
  epoch 1 | train_loss=0.7729 | val_auc=0.4766 | val_f1=0.6078 | t*=0.10
  epoch 2 | train_loss=0.7428 | val_auc=0.5242 | val_f1=0.6190 | t*=0.53
  epoch 3 | train_loss=0.7176 | val_auc=0.5565 | val_f1=0.6234 | t*=0.55
  epoch 4 | train_loss=0.7009 | val_auc=0.5831 | val_f1=0.6139 | t*=0.40
  epoch 5 | train_loss=0.6621 | val_auc=0.5677 | val_f1=0.6250 | t*=0.45
  epoch 6 | train_loss=0.6251 | val_auc=0.5363 | val_f1=0.6105 | t*=0.35

[2/3888] cfg={'d_model': 64, 'nhead': 2, 'ff_dim': 128, 'num_layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 16, 'use_pos_weight': False, 'epochs': 50, 'patience': 2}
  epoch 1 | train_loss=0.6946 | val_auc=0.5363 | val_f1=0.6200 | t*=0.38
  epoch 2 | train_loss=0.6669 | val_auc=0.5363 | val_f1=0.6078 | t*=0.10
  epoch 3 | train_loss=0.6392 | val_a